<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/g4/comfyui/qwen-image-2.1-bf16-comfy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# qwen-image-2.1-bf16-comfy · ComfyUI · G4 (96 GB)

One authenticated OpenAI-compatible API for **ComfyUI** models on a **Google AI Pro / Colab Pro** GPU.
**This notebook installs only `comfyui`** and advertises only that engine's catalog ids.

- Model: **Qwen-Image-2.1 BF16 (ComfyUI; best quality, A100 / G4)** (`qwen-image-2.1-bf16-comfy`)
- Sibling notebooks: other runtimes, engines and models: `notebooks/README.md`
- Drive root: `MyDrive/colab_model_server/` (shared weights/builds; settings file is `settings/settings_comfyui.json`)
- Engine matrix: `notebook_code/ENGINE_MATRIX.md`

Runtime → Change runtime type → **G4 (96 GB)**, then Runtime → Run all.


## 1. Mount Drive and create the workspace

The hub's code is packed into this notebook (the collapsed cell below): nothing is downloaded from GitHub. Settings, models and engine builds are restored from Drive.


In [ ]:
# @title Hub code, packed in this notebook (run this cell; nothing is downloaded)
# Generated from the notebook_code package; unpacked to /content/notebook_code on this runtime.
import base64, io, os, shutil, sys, zipfile
HUB_DIR = "/content/notebook_code"
HUB_CODE_VERSION = "f125ac8c760f"
HUB_ZIP = """
...
"""
shutil.rmtree(HUB_DIR, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(HUB_ZIP))).extractall(HUB_DIR)
del HUB_ZIP
if HUB_DIR not in sys.path:
    sys.path.insert(0, HUB_DIR)
for _m in [m for m in sys.modules if m == "notebook_code" or m.startswith("notebook_code.")]:
    del sys.modules[_m]          # a re-run picks up this cell's code, not an older import
print("Hub code", HUB_CODE_VERSION, "unpacked to", HUB_DIR)


In [ ]:
import os, re, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

ENGINE   = "comfyui"  # locked for this notebook
DRIVE_DIR = "/content/drive/MyDrive/colab_model_server"
HUB_REF   = HUB_CODE_VERSION  # the packed code version (recorded in settings)
assert os.path.isdir(f"{HUB_DIR}/notebook_code"), "Run the hub code cell above first."
os.makedirs(DRIVE_DIR, exist_ok=True)

from notebook_code.workspace import open_workspace
from notebook_code import install
WS = open_workspace(DRIVE_DIR, engine=ENGINE)
print("Drive workspace:", WS.describe())
import notebook_code
print("hub", notebook_code.__version__, "(code", HUB_CODE_VERSION + ")", "engine", ENGINE)


## 2. Detect GPU, CPU, and RAM — then list this engine's models


In [ ]:
from notebook_code import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # GPU + CPU + RAM + disk, every run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ARCH = G["arch"]  # compute capability without the dot: 89 L4, 80 A100, 120 G4
RAM_GB = RAM["total_gb"]
from notebook_code.store import model_disk_gb
DISK_GB = model_disk_gb(WS.local_models)   # free space + cached local models (they are evictable)
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH)
print()
print(f"Models are read from Google Drive (cached on local disk as they load); SSD-streamed files "
      f"are copied to {WS.local_models} ({DISK_GB} GB usable). Fit checks use VRAM, RAM and that disk.")
print(recommend.explain(VRAM, ALL, engine=ENGINE, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH))
print()
print(f"{'model id':36s} {'kind':5s} {'size':>7s}  {'ctx/edge':>14s}")
for e in ENTRIES:
    reason = catalog.why_not(e, VRAM, RAM_GB, DISK_GB, ARCH)
    fit = reason is None
    kind = catalog.kind_for(e)
    label = "edge" if kind in ("image", "video") else "ctx"
    ctx = f"{catalog.context_for(e, VRAM):,} {label}" if fit else "-"
    mark = " *" if e["id"] in PLAN["prefetch"] else ""
    print(f"{e['id']:36s} {kind:5s} {e['size_gb']:6.1f}G  {ctx:>14s}{mark}" + ("" if fit else f"   (does not fit: {reason})"))
print("\n* = recommended for this GPU + engine.")


## 3. Config (saved on Drive under `settings/settings_comfyui.json`)

This notebook serves **`qwen-image-2.1-bf16-comfy`**: the only model this session downloads and serves. Other settings come from Drive; edit and re-run to persist.


In [ ]:
from notebook_code import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # re-detect GPU + CPU + RAM every config run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ARCH = G["arch"]  # compute capability without the dot: 89 L4, 80 A100, 120 G4
RAM_GB = RAM["total_gb"]
from notebook_code.store import model_disk_gb
DISK_GB = model_disk_gb(WS.local_models)   # free space + cached local models (they are evictable)
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH)
saved = WS.load_settings()

# The ONE model this session downloads to Drive and serves. Nothing else is fetched.
MODEL = "qwen-image-2.1-bf16-comfy"  # this notebook's model (notebooks/<runtime>/<engine>/<model>.ipynb)
if PLAN["band"] != "g4":
    print("Note: this notebook is meant for the G4 (96 GB) runtime; this one is", G["name"])
SERVE_ONLY_SELECTED = True  # @param {type:"boolean"}
fitting = [e["id"] for e in ENTRIES if catalog.fits(e, VRAM, RAM_GB, DISK_GB, ARCH)]
if MODEL == "recommended":
    MODEL = (PLAN["prefetch"] or fitting or [None])[0]
    print("Recommended model for this GPU:", MODEL)
assert MODEL, f"No {ENGINE} model fits this GPU ({VRAM} GB); try a larger runtime."
assert MODEL in {e["id"] for e in ENTRIES}, f"{MODEL} is not a {ENGINE} model"
assert MODEL in fitting, (f"{MODEL} does not fit this runtime: "
                         f"{catalog.why_not(catalog.by_id(ENTRIES)[MODEL], VRAM, RAM_GB, DISK_GB, ARCH)}")
PREFETCH = [MODEL]
ENGINES = [ENGINE]  # never install other engines from this notebook
CTX_OVERRIDE = saved.get("ctx_override") or {}
LLAMA_CPP_REF = saved.get("llama_cpp_ref") or "4c3c5810f87efe177949f1b85ae509d38a2a311f"
REBUILD_LLAMA = bool(saved.get("rebuild_llama", False))
REFRESH_WHEELS = bool(saved.get("refresh_wheels", False))
# The tunnel is chosen in cell 6 when it starts (a prompt there); this is only the last choice.
TUNNEL = saved.get("tunnel") or "tailscale"
CF_HOSTNAME = saved.get("cf_hostname") or ""  # cloudflare-named only, e.g. llm.example.com
REFRESH_CF_TOKEN = bool(saved.get("refresh_cloudflare_token", False))
PORT = int(saved.get("port") or 8000)
TS_HOSTNAME = saved.get("ts_hostname") or f"colab-{ENGINE}"
TS_FUNNEL = bool(saved.get("ts_funnel", False))  # False = tailnet-only Serve
REFRESH_TS_KEY = bool(saved.get("refresh_tailscale_key", False))

WS.save_settings({
    "engine": ENGINE, "model": MODEL, "serve_only_selected": SERVE_ONLY_SELECTED,
    "engines": ENGINES,
    "ctx_override": CTX_OVERRIDE, "llama_cpp_ref": LLAMA_CPP_REF,
    "rebuild_llama": REBUILD_LLAMA, "refresh_wheels": REFRESH_WHEELS,
    "tunnel": TUNNEL, "port": PORT, "hub_ref": HUB_REF,
    "ts_hostname": TS_HOSTNAME, "ts_funnel": TS_FUNNEL, "cf_hostname": CF_HOSTNAME,
    "refresh_cloudflare_token": False,
    "refresh_tailscale_key": False,  # set True once to re-prompt and overwrite Drive key
    "gpu_name": G["name"], "vram_gb": VRAM, "band": PLAN["band"],
    "cpu_model": CPU["model"], "cpu_cores": CPU["cores"], "cpu_threads": CPU["threads"],
    "ram_gb": RAM["total_gb"], "ram_available_gb": RAM["available_gb"],
})
print("Saved settings →", WS.settings_path)
print("MODEL    =", MODEL, "(the only model downloaded and served)" if SERVE_ONLY_SELECTED else "(prefetched; others load on request)")
print("ENGINE   =", ENGINE)
print("Tailscale key file (private Drive, not git):", WS.hub / "tailscale_authkey.txt")


## 4. Install ComfyUI (built or downloaded once, then restored from Drive)


In [ ]:
from notebook_code import install

BUILDS = str(WS.builds)
LLAMA_BIN = "llama-server"
STRATA_BIN = "strata"
MAYA_ROOT = "/content/project-maya"
TENSORFOLD_BIN = "tensorfold-native"
COMFYUI_ROOT = "/content/ComfyUI"
SPARSA_ROOT = "/content/Sparsa"
HYPERQWEN_ROOT = install.HYPERQWEN_ROOT
NINFER_BIN = install.NINFER_ROOT + "/apps/ninfer-serve"
BASALT_ROOT = install.BASALT_ROOT
STRATA_NVFP4_ROOT = install.STRATA_NVFP4_ROOT
install.ensure_hub_deps(BUILDS, refresh=REFRESH_WHEELS)
if ENGINE == "llama":
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
elif ENGINE == "vllm":
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "sglang":
    print("SGLang venv python:", install.ensure_sglang(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "diffusers":
    print("diffusers", install.ensure_diffusers(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "strata":
    STRATA_BIN = install.ensure_strata(BUILDS, refresh=REFRESH_WHEELS)
    print("strata:", STRATA_BIN)
elif ENGINE == "maya":
    MAYA_ROOT = install.ensure_maya(BUILDS, refresh=REFRESH_WHEELS)
    print("maya:", MAYA_ROOT)
elif ENGINE == "tensorfold":
    TENSORFOLD_BIN = install.ensure_tensorfold(BUILDS, refresh=REFRESH_WHEELS)
    print("tensorfold:", TENSORFOLD_BIN)
elif ENGINE == "comfyui":
    COMFYUI_ROOT = install.ensure_comfyui(BUILDS, refresh=REFRESH_WHEELS)
    print("comfyui:", COMFYUI_ROOT)
elif ENGINE == "sparsa":
    SPARSA_ROOT = install.ensure_sparsa(BUILDS, refresh=REFRESH_WHEELS)
    print("sparsa:", SPARSA_ROOT)
elif ENGINE == "hyperqwen":
    HYPERQWEN_ROOT = install.ensure_hyperqwen(BUILDS, refresh=REFRESH_WHEELS)
    print("hyperqwen:", HYPERQWEN_ROOT)
elif ENGINE == "ninfer":
    NINFER_BIN = install.ensure_ninfer(BUILDS, G["arch"], refresh=REFRESH_WHEELS)
    print("ninfer-serve:", NINFER_BIN)
elif ENGINE == "basalt":
    BASALT_ROOT = install.ensure_basalt(BUILDS, G["arch"], refresh=REFRESH_WHEELS)
    print("basalt:", BASALT_ROOT)
elif ENGINE == "strata_nvfp4":
    STRATA_NVFP4_ROOT = install.ensure_strata_nvfp4(BUILDS, G["arch"], refresh=REFRESH_WHEELS)
    print("strata-nvfp4:", STRATA_NVFP4_ROOT)
print("Install log →", WS.logs / "install.log")


## 5. Download the models to Google Drive (skipped for anything already saved)

To spend less GPU time, run `notebooks/cpu/prepare_cpu.ipynb` on a CPU runtime first: it downloads the model and builds this engine on Drive, and this cell and cell 4 then just restore them.


In [ ]:
import threading, time
from notebook_code.store import ModelStore

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, str(WS.local_models))
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report(mid=mid, e=e):
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")


## 6. Start the hub and open the tunnel

Re-running this cell restarts the hub. Secrets stay on **your private Google Drive** (`MyDrive/colab_model_server/access/`), never in this public GitHub repo or in settings JSON:

- `access/api_key` — hub Bearer token (auto-created; `access/api_key.txt` is the same key, viewable on a phone)
- `access/tailscale_authkey.txt` — Tailscale auth key (asked once, then reused; open or edit it in the Drive app)
- `access/cloudflare_tunnel_token.txt` — only for the named Cloudflare tunnel (asked once, then reused)

**Tailscale** (default): create a **reusable + ephemeral** key at [Tailscale Keys](https://login.tailscale.com/admin/settings/keys) and join the machine your harness runs on to the same tailnet.

**Keys, the easy way: Colab Secrets.** Click the key icon in Colab's left sidebar and add `TAILSCALE_AUTHKEY` (and `HF_TOKEN`, and `CLOUDFLARE_TUNNEL_TOKEN` for a named Cloudflare tunnel). They belong to your Google account, so every notebook uses them; allow this notebook access once when Colab asks. A secret wins over the copy on Drive.

**When this cell runs it asks two things in its output, so you never edit code:** which tunnel (type 1, 2 or 3; Enter keeps the last one), then any key or token Drive does not have yet, in a hidden input box. Type e.g. `1 new` at the tunnel prompt to replace a saved key.

**Cloudflare quick tunnel** (choice 2): nothing to set up, like Unsloth Studio's notebook. The URL is public and changes every session, so update your harness each time; the hub's API key is what keeps others off your GPU. The named tunnel (choice 3) uses one you created in the Cloudflare dashboard (public hostname → `http://127.0.0.1:8000`) for a stable URL.

Your harness runs on **your own machine**, not here: point it at the Base URL this cell prints.


In [ ]:
import json, requests, time
from notebook_code import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[n]otebook_code serve'; pkill -TERM -f '[t]ailscaled.*colab-llm-tailscale'", shell=True)
time.sleep(3)

API_KEY = keyfile.load_or_create(str(WS.hub / "api_key"))
HUB_LOG = str(WS.logs / f"hub_{ENGINE}.log")
args = [sys.executable, "-m", "notebook_code", "serve", "--drive-dir", DRIVE_DIR,
        "--local-dir", str(WS.local_models), "--log-dir", str(WS.engine_logs),
        "--port", str(PORT), "--llama-bin", LLAMA_BIN,
        "--strata-bin", STRATA_BIN, "--maya-root", MAYA_ROOT,
        "--tensorfold-bin", TENSORFOLD_BIN, "--comfyui-root", COMFYUI_ROOT,
        "--sparsa-root", SPARSA_ROOT,
        "--hyperqwen-root", HYPERQWEN_ROOT, "--ninfer-bin", NINFER_BIN,
        "--basalt-root", BASALT_ROOT, "--strata-nvfp4-root", STRATA_NVFP4_ROOT,
        "--engines", ENGINE]
if SERVE_ONLY_SELECTED:
    args += ["--models", MODEL]   # the harness cannot trigger downloads of other models
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR, COLAB_LLM_DRIVE=DRIVE_DIR),
                            stdout=open(HUB_LOG, "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
LAST = "no answer yet"
for i in range(90):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=5,
                         proxies={"http": None, "https": None})
        if r.status_code == 200:
            print("Hub is up.")
            break
        LAST = f"HTTP {r.status_code}: {r.text[:200]}"
    except requests.RequestException as e:
        LAST = f"{type(e).__name__}: {str(e)[:200]}"
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open(HUB_LOG).read()[-3000:])
    if i in (10, 30, 60):
        print(f"Still waiting for the hub ({i} s); last answer: {LAST}")
    time.sleep(1)
else:
    raise RuntimeError(f"The hub did not answer /health (last: {LAST}):\n" + open(HUB_LOG).read()[-3000:])

# Pick the tunnel in the prompt below (Enter = the last one). Keys and tokens are asked for in a hidden
# input box when Drive has none yet, then kept on private Drive (access/), never in this notebook or git.
TUNNEL, NEW_KEY = tunnel.choose_tunnel(saved.get("tunnel"))
if TUNNEL == "cloudflare-named":
    CF_HOSTNAME = tunnel.ask_hostname(CF_HOSTNAME or None)
WS.save_settings({**WS.load_settings(), "tunnel": TUNNEL, "cf_hostname": CF_HOSTNAME})
if TUNNEL == "tailscale":
    TS_AUTHKEY = tunnel.load_or_ask_tailscale_authkey(str(WS.hub / "tailscale_authkey"),
                                                      refresh=REFRESH_TS_KEY or NEW_KEY)
    tunnel_proc, BASE = tunnel.start_tailscale(
        PORT, TS_AUTHKEY, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
    del TS_AUTHKEY  # drop from notebook namespace; Drive file remains private
elif TUNNEL == "cloudflare":
    tunnel_proc, BASE = tunnel.start_cloudflare_quick(PORT, drive_dir=DRIVE_DIR, api_key=API_KEY)
elif TUNNEL == "cloudflare-named":
    CF_TOKEN = tunnel.load_or_ask_secret(
        str(WS.hub / "cloudflare_tunnel_token"), "Cloudflare tunnel token (Zero Trust → Networks → "
        "Tunnels → your tunnel → install command): ", refresh=REFRESH_CF_TOKEN or NEW_KEY,
        env_name="CLOUDFLARE_TUNNEL_TOKEN")
    tunnel_proc, BASE = tunnel.start_cloudflare_named(PORT, CF_TOKEN, CF_HOSTNAME, drive_dir=DRIVE_DIR,
                                                      api_key=API_KEY)
    del CF_TOKEN
else:
    raise ValueError(f"Unknown TUNNEL {TUNNEL!r}")

# Serving from here on: release this GPU runtime after this many minutes with no API request,
# no cell running and no GPU / CPU / network work, so it stops using compute units (0 = never).
# Setup (installs, downloads) is never timed out: the clock only starts now.
IDLE_TIMEOUT_MIN = 10  # @param {type:"integer"}
from notebook_code import idle_guard
if idle_guard.start(IDLE_TIMEOUT_MIN):
    print(f"Idle timeout: this runtime is released after {IDLE_TIMEOUT_MIN} idle minutes.")

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
WS.save_endpoint(BASE + "/v1", API_KEY, G["name"], [m["id"] for m in MODELS])

print("Engine   :", ENGINE)
print("Tunnel   :", TUNNEL)
print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print("Endpoint →", WS.hub / "endpoint.json")
if TUNNEL == "tailscale" and not TS_FUNNEL:
    print("Reach this URL from any device on your Tailscale tailnet (e.g. the machine your harness runs on).")
elif TUNNEL == "cloudflare":
    print("Public URL, new every session: update your harness's base URL each time. The API key protects it.")
print()
# Your Mac only needs these settings: nothing to install or run there besides your harness.
DSH_YAML = provider.dsh_provider_yaml(BASE, MODELS, provider_id=f"colab-{ENGINE}")
(WS.hub / f"dsh_{ENGINE}.yaml").write_text(DSH_YAML)
print("=== dsh on your Mac ===")
print("1. Add this provider to your dsh settings.yaml (if the file already has 'llm-pi-ai:' / 'providers:',")
print(f"   paste from the 'colab-{ENGINE}:' line down, keeping the indentation). Also saved to Drive: "
      f"colab_model_server/access/dsh_{ENGINE}.yaml")
print()
print(DSH_YAML)
print("2. Give dsh the key (it is never written into settings.yaml), e.g. in ~/.zshrc:")
print(f"   export COLAB_LLM_API_KEY='{API_KEY}'")
print("3. Restart dsh and pick the provider. The URL stays the same every session, so this is one-time.")
print()
print("=== Any other OpenAI-compatible client ===")
print(provider.openai_env(BASE, API_KEY))


## 7. Try image generation

The first request loads the pipeline (Drive→local copy, then GPU). Non-streaming may return `503` + `Retry-After` while loading — re-run the cell.


In [ ]:
from IPython.display import display, Image as IPyImage
import base64

by = catalog.by_id(ENTRIES)
image_models = [m for m in MODELS if m["id"] in by and catalog.kind_for(by[m["id"]]) == "image"]
if not image_models:
    print("No image model picked; skipping.")
else:

    IMG = next((m for m in PREFETCH if m in {x["id"] for x in image_models}), image_models[0]["id"])
    prompt = "a watercolor fox under pine trees, soft light"
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/images/generations", headers=AUTH, timeout=900,
                      json={"model": IMG, "prompt": prompt, "size": "768x768", "n": 1})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Image model still loading; re-run this cell in a minute.")
    else:
        j = r.json()
        print("=== image:", IMG, j.get("size"), "===")
        if r.ok:
            display(IPyImage(data=base64.b64decode(j["data"][0]["b64_json"])))
            path = WS.save_image_b64(j["data"][0]["b64_json"], IMG, prompt=prompt)
            print(f"[{time.time() - t0:.1f} s] Saved →", path)
        else:
            print(j)


## 7b. Try video generation (MiniMax H3)

Runs only when a video model is picked. A 5 s clip takes minutes (longer on an L4); the MP4 is saved to Drive (`outputs/images`).


In [ ]:
from IPython.display import Video
video_models = [m for m in MODELS if m["id"] in by and catalog.kind_for(by[m["id"]]) == "video"]
if not video_models:
    print("No video model picked; skipping.")
else:
    VID = next((m for m in PREFETCH if m in {x["id"] for x in video_models}), video_models[0]["id"])
    vprompt = "a red fox trotting through fresh snow at dawn, soft light, natural sound"
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/videos/generations", headers=AUTH, timeout=7200,
                      json={"model": VID, "prompt": vprompt, "size": "832x480", "seconds": 5})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Video model still loading; re-run this cell in a minute.")
    elif r.ok:
        j = r.json()
        path = WS.save_video_b64(j["data"][0]["b64_mp4"], VID, prompt=vprompt)
        print(f"[{time.time() - t0:.0f} s] {j.get('size')}, {j.get('seconds')} s → {path}")
        display(Video(str(path), embed=True, width=640))
    else:
        print(r.status_code, r.text[:1000])


## 8. Status and logs (re-run any time)


In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
WS.save_json(WS.status, f"hub-status-{ENGINE}", st)
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log (Drive) ---"); print(open(HUB_LOG).read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print("Outputs →", WS.outputs)


## 9. Stop


In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
try:
    from notebook_code import dsh_web
    dsh_web.stop()            # the in-notebook harness, if it was started
except Exception:
    pass
try:
    tunnel_proc.terminate()
except Exception:
    pass
if TUNNEL == "tailscale":
    tunnel.stop_tailscale(drive_dir=DRIVE_DIR)
else:
    tunnel.stop_cloudflare()


## Notes

### This notebook = `comfyui` only

| Catalog id | Notes |
|---|---|
| `qwen-image-2.1-turbo-comfy` | Qwen-Image 2.1 Turbo INT8 + INT8 encoder (L4, 8 steps) |
| `qwen-image-2.1-turbo-bf16-comfy` | Turbo, full precision (A100 / G4, 8 steps) |
| `qwen-image-2.1-bf16-comfy` | Base model, full precision (A100 / G4, 25 steps, best images) |
| `flux3-klein-comfy` | FLUX.3 Klein |
| `minimax-h3-l4-comfy` | MiniMax H3 video + audio: pruned INT8 (L4, slow, short 480p clips) |
| `minimax-h3-comfy` | MiniMax H3: INT8 + INT8 encoder (A100) |
| `minimax-h3-g4-comfy` | MiniMax H3: INT8 + NVFP4 encoder (G4) |
| `wan22-5b-comfy` | Wan 2.2 5B (prefetch; UI workflow only) |

Video: `POST /v1/videos/generations` with `prompt`, `size` (e.g. `832x480`) and `seconds` (default 5); the MP4 comes back as `b64_mp4`.

UI graphs under `notebook_code/workflows/*.ui.json` (custom NvidiaResident nodes are optional).

### Connecting your harness

**Tailscale** (choice 1, default, private):

1. Create a **reusable + ephemeral** auth key: https://login.tailscale.com/admin/settings/keys
2. First run: cell 6 asks for it in a hidden input box and saves it to **private Drive** `MyDrive/colab_model_server/access/tailscale_authkey.txt` (not GitHub).
3. Later Colab sessions reuse that Drive file automatically.
4. Easier: add it once as the Colab secret `TAILSCALE_AUTHKEY` (key icon in the left sidebar); every notebook then uses it without asking.
5. Join the machine your harness runs on to the same tailnet and point the harness at the printed `https://….ts.net/v1` URL with the API key. The harness (dsh or any OpenAI client) runs there, never in this notebook.

**Cloudflare quick tunnel** (choice 2): no account or key. Cell 6 prints a public `https://….trycloudflare.com/v1` URL once the hub answers through it; it changes every session, and anyone with it reaches the hub, which still requires the API key. Quick tunnels are a free best-effort Cloudflare service with no uptime guarantee.

**Cloudflare named tunnel** (choice 3): create a tunnel in the Cloudflare dashboard (Zero Trust → Networks → Tunnels), add a public hostname pointing to `http://127.0.0.1:8000`; cell 6 then asks for the hostname and, once, the tunnel token (saved to `access/cloudflare_tunnel_token` on private Drive).

### Drive layout

| Path under `MyDrive/colab_model_server` | Contents |
|---|---|
| `models/<id>/` | Weights shared across engine notebooks |
| `engines/` | llama.cpp tarballs, wheels, Tailscale, cloudflared |
| `settings/settings_comfyui.json` | Selected model / ctx / Tailscale for **this** engine (no secrets) |
| `access/api_key`, `access/api_key.txt` | Hub Bearer token (private Drive only; the .txt copy opens on phones) |
| `access/tailscale_authkey.txt` | Tailscale auth key (private Drive only; never in git; opens on phones) |
| `access/cloudflare_tunnel_token.txt` | Cloudflare named-tunnel token (private Drive only; never in git) |
| `access/endpoint.json` | Last base URL + key hint |
| `huggingface/`, `logs/`, `outputs/` | Cache, logs, generation results |

- See `notebook_code/ENGINE_MATRIX.md` for workstation (Maya / Strata / HyperQwen) notes.
- Set `REFRESH_WHEELS` / `REBUILD_LLAMA` / `refresh_tailscale_key` in settings to force updates.
- **Security:** Tailscale Serve is reachable only from your tailnet (`ts_funnel` makes it public); Cloudflare URLs are public. Either way the hub API key protects GPU use. Never commit anything under `access/`.
